##  Build a model using transfer learning with a pretrained VGG network as the base. Add custom layers on top, including dense layers that incorporate additional features (e.g., business or photo metadata), and fine-tune the entire model.`

In [1]:
import os

train_dir = './train'  
test_dir = './test'    
meta_csv = './balanced_photos_5000/balanced_photos_metadata.csv'   

 Import Metadata and Prepare Lookup

In [6]:
import tensorflow as tf
print("TensorFlow version:", tf.__version__)
print("Built with CUDA:", tf.test.is_built_with_cuda())
print("GPUs Available:", tf.config.list_physical_devices('GPU'))


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "C:\Users\User\anaconda3\envs\tf210gpu\lib\runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\User\anaconda3\envs\tf210gpu\lib\runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "C:\Users\User\anaconda3\envs\tf210gpu\lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\User\anaconda3\envs\tf210gpu\lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.s

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "C:\Users\User\anaconda3\envs\tf210gpu\lib\runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\User\anaconda3\envs\tf210gpu\lib\runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "C:\Users\User\anaconda3\envs\tf210gpu\lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\User\anaconda3\envs\tf210gpu\lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.s

AttributeError: _ARRAY_API not found

ImportError: numpy.core._multiarray_umath failed to import

ImportError: numpy.core.umath failed to import

TypeError: Unable to convert function return value to a Python type! The signature was
	() -> handle

In [5]:
import pandas as pd

meta_df = pd.read_csv(meta_csv)

meta_df['filename'] = meta_df['photo_id'].astype(str) + '.jpg'

ImportError: C extension: pandas.util not built. If you want to import pandas from the source directory, you may need to run 'python setup.py build_ext' to build the C extensions first.

In [5]:
from sklearn.preprocessing import LabelEncoder

if 'business_id_encoded' not in meta_df.columns:
    lbl_enc = LabelEncoder()
    meta_df['business_id_encoded'] = lbl_enc.fit_transform(meta_df['business_id'].astype(str))

In [6]:
print(meta_df.columns)

Index(['photo_id', 'business_id', 'caption', 'label', 'filename',
       'business_id_encoded'],
      dtype='object')


In [7]:
if 'caption_length' not in meta_df.columns:
    meta_df['caption_length'] = meta_df['caption'].fillna('').apply(len)

Gather Image Paths, Metadata, and Labels

In [9]:
import numpy as np

def gather_data(folder_dir, meta_df):
    class_names = sorted([d for d in os.listdir(folder_dir) if os.path.isdir(os.path.join(folder_dir, d))])
    class_to_idx = {cls: i for i, cls in enumerate(class_names)}
    img_paths, meta_features, labels = [], [], []

    for cls in class_names:
        cls_dir = os.path.join(folder_dir, cls)
        for fname in os.listdir(cls_dir):
            if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                img_paths.append(os.path.join(cls_dir, fname))
                labels.append(class_to_idx[cls])
                row = meta_df[meta_df['filename'] == fname]
                if not row.empty:
                    meta_features.append([
                        row.iloc[0]['business_id_encoded'],
                        row.iloc[0]['caption_length']
                    ])
                else:
                    meta_features.append([0, 0])  
    return np.array(img_paths), np.array(meta_features), np.array(labels), class_names

X_train_img, X_train_meta, y_train, class_names = gather_data(train_dir, meta_df)
X_test_img,  X_test_meta,  y_test,  _          = gather_data(test_dir,  meta_df)
num_classes = len(class_names)

print("Train size:", len(X_train_img))
print("Test size:", len(X_test_img))
print("Classes:", class_names)

Train size: 20000
Test size: 5000
Classes: ['drink', 'food', 'inside', 'menu', 'outside']


In [10]:
def build_df(img_paths, meta_features, labels):
    return pd.DataFrame({
        'img_path': img_paths,
        'business_id_encoded': meta_features[:, 0],
        'caption_length': meta_features[:, 1],
        'label': labels
    })

train_df = build_df(X_train_img, X_train_meta, y_train)
test_df  = build_df(X_test_img,  X_test_meta,  y_test)

In [11]:
print("Train DataFrame size:", len(train_df))
print("Test DataFrame size:", len(test_df))
print(train_df.head())

Train DataFrame size: 20000
Test DataFrame size: 5000
                                          img_path  business_id_encoded  \
0         ./train\drink\-0-Sor-tVeKH_KXSWI8SMw.jpg                 1315   
1    ./train\drink\-0-Sor-tVeKH_KXSWI8SMw_blur.jpg                    0   
2  ./train\drink\-0-Sor-tVeKH_KXSWI8SMw_thresh.jpg                    0   
3         ./train\drink\-3u457qLcpIs6CYgGSgpeA.jpg                 4666   
4    ./train\drink\-3u457qLcpIs6CYgGSgpeA_hist.jpg                    0   

   caption_length  label  
0              20      0  
1               0      0  
2               0      0  
3               0      0  
4               0      0  


Create tf.data Datasets

In [14]:
import tensorflow as tf

num_classes = len(train_df['label'].unique())

def parse_row(img_path, business_id_encoded, caption_length, label):
    img = tf.io.read_file(img_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [224, 224])
    img = img / 255.0
    meta = tf.stack([
        tf.cast(business_id_encoded, tf.float32),
        tf.cast(caption_length, tf.float32)
    ])
    label = tf.one_hot(label, num_classes)
    return (img, meta), label

def build_tf_dataset(df, batch_size=32, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices((
        df['img_path'].values,
        df['business_id_encoded'].values,
        df['caption_length'].values,
        df['label'].values
    ))
    ds = ds.map(parse_row, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(buffer_size=len(df))
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

batch_size = 32
train_ds = build_tf_dataset(train_df, batch_size=batch_size, shuffle=True)
test_ds  = build_tf_dataset(test_df,  batch_size=batch_size, shuffle=False)

Build Multi-Input VGG19 + Metadata Model

In [29]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.layers import Input, Dense, Dropout, Flatten, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import tensorflow as tf

callbacks = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', patience=2, factor=0.5)
]

n_metadata_features = 2
num_classes = len(train_df['label'].unique()) # Assuming train_df is defined

img_input = Input(shape=(224, 224, 3), name='img_input')
meta_input = Input(shape=(n_metadata_features,), name='meta_input')

vgg = VGG16(weights='imagenet', include_top=False, input_tensor=img_input)

# Freeze the VGG base model layers
vgg.trainable = False

x = Flatten()(vgg.output)
y = Dense(32, activation='relu')(meta_input)
combined = Concatenate()([x, y])
z = Dense(128, activation='relu')(combined)
z = Dropout(0.3)(z)
output = Dense(num_classes, activation='softmax')(z)

model = Model(inputs=[img_input, meta_input], outputs=output)

# Compile and train the model with frozen VGG layers
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

# Now train the model for a few epochs
print("\n--- Training with VGG layers frozen ---")
history_frozen = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=5, 
    callbacks=callbacks 
)

# Unfreeze VGG layers for fine-tuning 
print("\n--- Unfreezing VGG layers for fine-tuning ---")
vgg.trainable = True


# Using a very low learning rate for fine-tuning
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

# Continue training
history_fine_tune = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=15, 
    callbacks=callbacks
)

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ img_input (InputLayer)        │ (None, 224, 224, 3)       │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block1_conv1 (Conv2D)         │ (None, 224, 224, 64)      │           1,792 │ img_input[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block1_conv2 (Conv2D)         │ (None, 224, 224, 64)      │          36,928 │ block1_conv1[0][0]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block1_pool (MaxPooling2D)    │ (None, 112, 112, 64)      │               0 │ block1_conv2[0][0]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block2_conv1 (Conv2D)         │ (None, 112, 112, 128)     │          73,856 │ block1_pool[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block2_conv2 (Conv2D)         │ (None, 112, 112, 128)     │         147,584 │ block2_conv1[0][0]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block2_pool (MaxPooling2D)    │ (None, 56, 56, 128)       │               0 │ block2_conv2[0][0]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block3_conv1 (Conv2D)         │ (None, 56, 56, 256)       │         295,168 │ block2_pool[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block3_conv2 (Conv2D)         │ (None, 56, 56, 256)       │         590,080 │ block3_conv1[0][0]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block3_conv3 (Conv2D)         │ (None, 56, 56, 256)       │         590,080 │ block3_conv2[0][0]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block3_pool (MaxPooling2D)    │ (None, 28, 28, 256)       │               0 │ block3_conv3[0][0]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block4_conv1 (Conv2D)         │ (None, 28, 28, 512)       │       1,180,160 │ block3_pool[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block4_conv2 (Conv2D)         │ (None, 28, 28, 512)       │       2,359,808 │ block4_conv1[0][0]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block4_conv3 (Conv2D)         │ (None, 28, 28, 512)       │       2,359,808 │ block4_conv2[0][0]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block4_pool (MaxPooling2D)    │ (None, 14, 14, 512)       │               0 │ block4_conv3[0][0]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block5_conv1 (Conv2D)         │ (None, 14, 14, 512)       │       2,359,808 │ block4_pool[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block5_conv2 (Conv2D)         │ (None, 14, 14, 512)       │       2,359,808 │ block5_conv1[0][0]         │
├───────────────────────────────┼───────────────────────────┼───────────────

 Total params: 17,930,917 (68.40 MB)

 Trainable params: 3,216,229 (12.27 MB)

 Non-trainable params: 14,714,688 (56.13 MB)


--- Training with VGG layers frozen ---
Epoch 1/5
 29/625 ━━━━━━━━━━━━━━━━━━━━ 1:23:23 8s/step - accuracy: 0.3572 - loss: 7.4797

KeyboardInterrupt: 